In [ ]:
import json
import pickle
import re
from collections import defaultdict

import faiss
import numpy as np
import pandas as pd
from nltk.stem import WordNetLemmatizer
from sentence_transformers import CrossEncoder, SentenceTransformer
from rank_bm25 import BM25Okapi

In [ ]:
train = pd.read_parquet("../data/processed/corpora.parquet")
test = pd.read_parquet("../data/processed/evaluation.parquet")
catalogue = pd.read_parquet("../data/interim/catalogue.parquet")

metadata = train[["chapter", "heading", "subheading", "hkhs"]] \
    .to_dict("records")

#### Sparse Embedding Search

In [ ]:
class SparseSearchEngine:
    def __init__(self):
        self.model = None
        self.corpora, self.metadata = [], []

    def _tokenise(self, text):
        return re.findall(r"\w+", text.lower())

    def index(self, corpora, metadata):
        self.corpora = corpora.tolist()
        self.metadata = metadata

        tokens = [self._tokenise(corpus) for corpus in corpora]
        self.model = BM25Okapi(tokens)

    def _search_docs(self, queries):
        tokens = [self._tokenise(query) for query in queries.tolist()]

        return [self.model.get_scores(query) for query in tokens]

    def _score(self, scores, pool, decay):
        top_scores = sorted(scores, reverse=True)[:pool]

        return sum(score*decay**i for i, score in enumerate(top_scores))

    def search(self, queries, tag, k=10, depth=100, pool=4, decay=0.5):
        scores = self._search_docs(queries)
        
        predictions = []
        for score in scores:
            d = min(depth, len(score))
            top = np.argpartition(-score, d - 1)[:d]
            top = np.sort(top)[np.argsort(-score[np.sort(top)], kind="stable")]

            grouped = defaultdict(list)
            for i in top:
                if score[i] > 0:
                    grouped[self.metadata[i][tag]].append(score[i])

            ranked = sorted(
                grouped,
                key=lambda t: self._score(grouped[t], pool=pool, decay=decay),
                reverse=True,
            )[:k]

            predictions.append(ranked)

        return predictions

    def save(self, path):
        with open(f"{path}.pkl", "wb") as f:
            pickle.dump(self.model, f)

        with open(f"{path}.meta", "w") as f:
            json.dump({"corpora": self.corpora, "metadata": self.metadata}, f)

    def load(self, path):
        with open(f"{path}.pkl", "rb") as f:
            self.model = pickle.load(f)

        with open(f"{path}.meta", "r") as f:
            meta = json.load(f)
            self.corpora = meta["corpora"]
            self.metadata = meta["metadata"]

In [ ]:
source = ["desc", "desc_stem", "desc_lemma"][2]  # lemmatisation helps the most

model = SparseSearchEngine()
model.index(train[source], metadata)

predictions = {}
for tag in ["chapter", "heading", "subheading"]:
    predictions[tag] = model.search(test[source], tag)

%run ./05_search_evaluation.ipynb

In [ ]:
model.save("../checkpoints/sparse/bm25")

#### Dense Embedding Search

In [ ]:
class DenseSearchEngine:
    def __init__(self, model_card="BAAI/bge-small-en-v1.5", runtime=None):
        args = {"model_name_or_path": model_card}
        if runtime == "onnx":
            args["model_name_or_path"] += "-onnx"
            args.update({
                "backend": "onnx",
                "model_kwargs": {"provider": "CPUExecutionProvider"},
            })
        self.model = SentenceTransformer(**args)
        self.vector_index = faiss.IndexFlatIP(
            self.model.get_embedding_dimension()
        )
        self.corpora, self.metadata = [], []

    def _embed(self, corpora):
        vectors = self.model.encode(corpora.tolist(), convert_to_numpy=True)
        faiss.normalize_L2(vectors)

        return vectors

    def index(self, corpora, metadata):
        self.corpora = corpora.tolist()
        self.metadata = metadata

        vectors = self._embed(corpora)
        self.vector_index.add(vectors)

    def _search_docs(self, queries, depth):
        vectors = self._embed(queries)

        return self.vector_index.search(vectors, depth)

    def _score(self, scores, pool, decay):
        top_scores = sorted(scores, reverse=True)[:pool]

        return sum(score*decay**i for i, score in enumerate(top_scores))

    def search(self, queries, tag, k=10, depth=100, pool=4, decay=0.5):
        scores, indices = self._search_docs(queries, depth)
        predictions = []
        for score, index in zip(scores, indices):
            mask = index != -1
            floor = min(score[mask], default=0.0)

            grouped = defaultdict(list)
            for s, idx in zip(score[mask], index[mask]):
                grouped[self.metadata[idx][tag]].append(s - floor)

            ranked = sorted(
                grouped,
                key=lambda x: self._score(grouped[x], pool=pool, decay=decay),
                reverse=True,
            )[:k]

            predictions.append(ranked)

        return predictions

    def save(self, path):
        faiss.write_index(self.vector_index, f"{path}.faiss")

        with open(f"{path}.meta", "w") as f:
            json.dump({"corpora": self.corpora, "metadata": self.metadata}, f)

    def load(self, path):
        self.vector_index = faiss.read_index(f"{path}.faiss")

        with open(f"{path}.meta", "r") as f:
            meta = json.load(f)
            self.corpora = meta["corpora"]
            self.metadata = meta["metadata"]


class Reranker:
    def __init__(self, model_card="cross-encoder/ms-marco-MiniLM-L6-v2"):
        self.model = CrossEncoder(model_card)

    def rerank(self, queries, candidates):
        results = []
        for query, candidate in zip(queries, candidates):
            pairs = []
            for cand in candidate:
                desc = catalogue.query("hsc == @cand")["desc"].values[0]
                pairs.append((query, desc))

            scores = self.model.predict(pairs)
            order = np.argsort(-scores, kind="stable")
            results.append(np.asarray(candidate)[order].tolist())

        return results

In [ ]:
source = ["desc", "desc_stem", "desc_lemma"][0]  # semantic less depends on root/base form
model_card = [
    "BAAI/bge-small-en-v1.5", "BAAI/bge-base-en-v1.5",
    "all-MiniLM-L6-v2", "intfloat/multilingual-e5-base",
    "mixedbread-ai/mxbai-embed-large-v1",
    "../checkpoints/bge-small-en-v1.5-ft-hs/v0.1",
    "../checkpoints/bge-small-en-v1.5-ft-hs/v0.2",
    "../checkpoints/bge-small-en-v1.5-ft-hs/v0.3",
    "../checkpoints/bge-small-en-v1.5-ft-hs/v0.4",
    "../checkpoints/bge-small-en-v1.5-ft-hs/v0.5",
][-3]  # bge-small trades speed for accuracy with bge-base

model = DenseSearchEngine(model_card)
model.index(train[source], metadata)

reranker = Reranker()  # rerank does not help much

predictions = {}
for tag in ["chapter", "heading", "subheading"]:
    prediction = model.search(test[source], tag)
    # prediction = reranker.rerank(test[source], prediction)
    predictions[tag] = reranker.rerank(test[source], prediction)

%run ./05_search_evaluation.ipynb

In [ ]:
model.save("../checkpoints/dense/bge")

#### Hybrid Search

In [ ]:
class HybridSearchEngine:
    def __init__(self, model_card="BAAI/bge-small-en-v1.5", runtime=None):
        self.sparse = SparseSearchEngine()
        self.dense = DenseSearchEngine(model_card, runtime=runtime)
        self.lemmatiser = WordNetLemmatizer()
        self.corpora, self.metadata = [], []

    def _clean_text(self, text):
        text = text.lower()
        text = re.sub(r"[^a-zA-Z\s]", " ", text)
        text = re.sub(r"\s+", " ", text)
        return text.strip()

    def _lemmatise_desc(self, text):
        return " ".join(
            self.lemmatiser.lemmatize(word)
            for word in self._clean_text(text).split()
        )

    def index(self, corpora, metadata):
        self.corpora = corpora.tolist()
        self.metadata = metadata

        self.sparse.index(corpora.apply(self._lemmatise_desc), metadata)
        self.dense.index(corpora, metadata)

    def _fuse(self, rankings, c, weights):
        weights = weights or [1.0] * len(rankings)

        scores = defaultdict(float)
        for w, ranking in zip(weights, rankings):
            for rank, t in enumerate(ranking, start=1):
                scores[t] += w / (c + rank)

        return sorted(scores, key=scores.get, reverse=True)

    def search(
        self,
        queries,
        tag,
        k=10,
        depth=50,
        candidates=50,
        c=10,
        weights=(1.0, 2.0),
    ):
        lemmatised = queries.apply(self._lemmatise_desc)

        sparse = self.sparse.search(lemmatised, tag, k=candidates, depth=depth)
        dense = self.dense.search(queries, tag, k=candidates, depth=depth)

        return [
            self._fuse([s, d], c=c, weights=weights)[:k]
            for s, d in zip(sparse, dense)
        ]

    def save(self, path):
        self.sparse.save(path)
        self.dense.save(path)

    def load(self, path):
        self.sparse.load(path)
        self.dense.load(path)

In [ ]:
model = HybridSearchEngine("../checkpoints/bge-small-en-v1.5-ft-hs/v0.3")
model.index(train["desc"], metadata)

predictions = {}
for tag in ["chapter", "heading", "subheading"]:
    predictions[tag] = model.search(test["desc"], tag)

%run ./05_search_evaluation.ipynb

In [ ]:
model.save("../checkpoints/hybrid/bm25_bge")

#### Playground

In [ ]:
sparse = SparseSearchEngine()
sparse.load("../checkpoints/sparse/bm25")
dense = DenseSearchEngine("../checkpoints/bge-small-en-v1.5-ft-hs/v0.3")
dense.load("../checkpoints/dense/bge")
hybrid = HybridSearchEngine("../checkpoints/bge-small-en-v1.5-ft-hs/v0.3")
hybrid.load("../checkpoints/hybrid/bm25_bge")

In [ ]:
word = "iphone duo"
print(sparse.search(pd.Series([word]), tag)[0])
print(dense.search(pd.Series([word]), tag)[0])
print(hybrid.search(pd.Series([word]), tag)[0])